# Módulos

In [ ]:
from pathlib import Path
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
import optuna
import xgboost as xgb
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import LabelBinarizer

# Leer Archivo

In [ ]:
p = Path(os.getcwd())
p = p.parents[2]
p1 = p / 'data' / 'processed' / 'precios.parquet'
p2 = p/ 'data' / 'processed' / 'popularidad.parquet'

df = pd.read_parquet(p1)
df.head(2)

# Procesar

## Drop de columnas

In [ ]:
df_clean = df.drop(columns=['id','name', 'price_overview', 'v_resnet' , 'v_convnext'])
df_clean.head(2)

# Modelo sin v_clip

In [ ]:
df_clean

## Procesar v_clip -> PCA

In [ ]:
df_clip = df['v_clip'].apply(pd.Series) 
pca = PCA(n_components=0.5)  
df_clip = pca.fit_transform(df_clip)
df_clip = pd.DataFrame(df_clip)
df_clip

In [ ]:
# Variance explained by each component
# As a DataFrame for better readability
variance_df = pd.DataFrame({
    'PC': [f'PC{i+1}' for i in range(len(pca.explained_variance_ratio_))],
    'Explained Variance': pca.explained_variance_ratio_,
    'Cumulative Variance': pca.explained_variance_ratio_.cumsum()
})
variance_df

Cada columna v_clip no da mucha información

In [ ]:
df_clip = df['v_clip'].apply(pd.Series) 
pca = PCA(n_components=0.8)  
df_clip = pca.fit_transform(df_clip)
df_clip = pd.DataFrame(df_clip)
df_clip

## Procesado final

In [ ]:
df = df_clean
df.drop(columns=['v_clip'], inplace=True, errors='ignore')
df = pd.concat([df, df_clip], axis=1)
df.columns = df.columns.astype(str)

lb = LabelBinarizer()
df['release_year'] = lb.fit_transform(df['release_year'])

df.head(2)

# División

In [ ]:
X = df.drop(columns=['price_range'])
y = df['price_range']

In [ ]:
lb = LabelBinarizer()
y  = lb.fit_transform(y)
y

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y,random_state=42, test_size=0.2)

In [ ]:
X_train.columns

In [ ]:
def objective(trial):
    params = {
        'max_depth' : trial.suggest_int('max_depth', 3, 10),
        'n_estimators' : trial.suggest_float('learning_rate', 0.01, 0.1),
        'learning_rate' : trial.suggest_int('n_estimators', 100, 1000),
        'subsample' : trial.suggest_float('subsample', 0.5, 1.0),
        'random_state' : 42
    }

    model = xgb.XGBClassifier(**params)
    score = cross_val_score(model, X_train, y_train, cv=3).mean() 
    return score

study = optuna.create_study(study_name="example_xgboost_study", direction='maximize') 
study.optimize(objective, n_trials=100, show_progress_bar=True, n_jobs=-1)

best_params = study.best_params
print(f"\nBest parameters: {best_params}")